In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")
model_name = os.getenv("MMISTRAL_MODEL")
print(f"Model Name: {model_name}")

In [ ]:
from utils.processor import docx2mark
from utils.web import generate_all_urls_for_keywords

resume = docx2mark("resume.docx")

In [ ]:
import pandas as pd
df_loaded = pd.read_csv('text/keywords.csv')
keywords = df_loaded["keyword"].tolist()

combined_df = generate_all_urls_for_keywords(keywords)
combined_df = combined_df.merge(
    df_loaded[["keyword", "category"]], 
    on="keyword", 
    how="left"
)


In [ ]:
API_BASE_URL = 'http://localhost:3000'
JOBS_API_ENDPOINT = f'{API_BASE_URL}/api/jobs'

print(f"Jobs API endpoint: {JOBS_API_ENDPOINT}")

In [ ]:
from utils.linkedin import fetch_jobs_from_df

all_jobs, failed_keywords = fetch_jobs_from_df(combined_df, JOBS_API_ENDPOINT)

In [ ]:
from utils.web import scrape_website

# Convert jobs to DataFrame
jobs_df = pd.DataFrame(all_jobs)

if not jobs_df.empty:
    # Select a job URL to test
    job_url = jobs_df.iloc[0]["jobUrl"]
    print(f"\nScraping job at: {job_url}")
    
    # Now using simple requests/BS4 fallback via the updated scrape_website
    job_content = await scrape_website(job_url)
    print(f"Scraped content length: {len(job_content)} characters")
    print("Successfully fetched page content.")
else:
    print("No jobs found to scrape.")

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(job_content,"html.parser") 
# This checks if "Job Description:" is anywhere inside the tag's text
target = soup.find(lambda tag: tag.name in ["strong", "span"] and "Job Description:" in tag.get_text())

if target:
    parent = target.parent
    print(f"Found parent: {parent.name} with class {parent.get('class')}")

In [ ]:
parent.getText()

In [ ]:
target.getText()